In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pylab as plt
import datetime
import scipy.stats
import plotly.offline as pyo
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.colors as cl
import matplotlib.dates as mdates
import matplotlib
import statsmodels.api as sm
from statsmodels.regression.rolling import RollingOLS
import seaborn as sns
import netCDF4 as nc
import xarray as xr
#import cartopy.crs as ccrs
#from cartopy import feature as cfa

In [ ]:
import plotly.io as pio
pio.renderers.default = 'notebook'

In [ ]:
%run ../../2025-12-20_AgeModel-NewXrayPatch/age_model/20260215_read_data.py
df_sorted

In [ ]:
%run ../../2025-08-22_FredsData-post2008/fred_data.py
fred

In [ ]:
def decimal_year_to_datetime(decimal_year):
    year = decimal_year.astype(int)
    decimal_part = decimal_year - year
    start_of_year = pd.to_datetime([f'{y}-01-01' for y in year])
    seconds_in_year = (pd.to_datetime([f'{y+1}-01-01' for y in year]) - start_of_year).total_seconds()
    seconds = decimal_part * seconds_in_year
    return start_of_year + pd.to_timedelta(seconds, unit='s')

def datetime_to_decimal_year(datetime_):
    year = datetime_.year
    days = (datetime_.date() - datetime.date(year, 1,1)).days
    res = year + days/365.25
    return res


def FFT_res(x, si):
    n = len(x)
    x_FFT_shifted = np.fft.fftshift(np.fft.fft(x))
    x_FFT_freq_shifted = np.fft.fftshift(np.fft.fftfreq(n, d=si))
    return x_FFT_freq_shifted, x_FFT_shifted


def fft_noramlize(fft_res):
    n = len(fft_res)
    norm_fft = np.abs(fft_res)/(np.max(np.abs(fft_res)))
    return norm_fft

def line_fit(X,Y):
    P = np.polyfit(X,Y,1)
    dx = np.max(X) - np.min(X)
    X_new = np.array([np.min(X)-dx*0.05, np.max(X)+dx*0.05])
    Y_new = np.polyval(P, X_new)
    return X_new, Y_new, P

In [ ]:
tide = pd.read_csv('../../2025-10-10_Fred-TideGauge/memo_edited.csv')
tide['age_dt'] = decimal_year_to_datetime(tide['Date'])#.dt.date
tide = tide.set_index('age_dt')
tide

In [ ]:
enso_phase = pd.read_csv('../../ENSO_Phase_Classification.csv')
enso_phase.columns=['Year','Phase']
enso_phase['Date'] = decimal_year_to_datetime(enso_phase['Year'])
#enso_phase.set_index('Date', inplace=True)
enso_phase['Phase'] = enso_phase['Phase'].str.replace('Warm Phase', 'W').str.replace('Cold Phase', 'C').str.replace('Neutral Phase','N')


enso = pd.read_csv("../../ENSO_Time_Series.csv", parse_dates=[0])
#enso = enso.set_index("Date")
enso['day_of_year'] = enso['Date'].dt.dayofyear
enso['is_leap'] = enso['Date'].dt.is_leap_year
enso['total_days'] = np.where(enso['is_leap'], 366, 365)
enso['decimal_year'] = enso['Date'].dt.year + (enso['day_of_year'] - 1) / enso['total_days']


margin = 0.5
enso['phase'] = 'N'
enso.loc[enso['ENSO']>=margin,'phase'] = "C"
enso.loc[enso['ENSO']<=-margin,'phase'] = "W"
enso['phase_group'] = (enso['phase'] != enso['phase'].shift()).cumsum()
blocks = enso.groupby('phase_group').agg(
    start=('Date', 'first'),
    end=('Date', 'last'),
    value=('phase', 'first')
).reset_index(drop=True)

ENSO_match = np.array([[1982.2, 1984.1],
                       [1986.75, 1988.3],
                       [1992.1, 1992.96],
                       [1997.2, 1998.8]])

enso_fft_freq, enso_fft = FFT_res(enso['ENSO'], 30)


enso


In [ ]:
#oisst = xr.open_dataset('../data/oisst2-weekly-156.875_157.125E-8.375_8.125S-19810906-20251019.nc')
#oisst = xr.open_dataset('../data/oisst2-monthly-156.875_157.125E-8.375_8.125S-198109-202509.nc')
oras5_sss_nc = xr.open_dataset('../data/oras5-sosaline-157E-8.5S-195801-201812.nc')
oras5_sst_nc = xr.open_dataset('../data/oras5-sosstsst-157E-8.5S-195801-201812.nc')
#oras5_sst_nc

In [ ]:
oras5_t_sst = oras5_sst_nc.mean(dim=['lat', 'lon'])['time'].values
oras5_sst = oras5_sst_nc.mean(dim=['lat', 'lon'])['sosstsst'].values

oras5_t = oras5_sss_nc.isel(lat=0, lon=0)['time'].values
oras5_sss = oras5_sss_nc.isel(lat=0, lon=0)['sosaline'].values

climate = pd.DataFrame([oras5_t, oras5_sst, oras5_sss]).T
climate = climate.rename(columns={0:'datetime', 1:'SST', 2:'SSS'})
climate['SST'] =  pd.to_numeric(climate['SST'])
climate['SSS'] =  pd.to_numeric(climate['SSS'])
climate['datetime'] = pd.to_datetime(climate['datetime'])
climate['age_absolute'] = climate['datetime'].apply(datetime_to_decimal_year)
climate.set_index('datetime', inplace=True)
climate

In [ ]:
MODEL = pd.read_csv('coral_signal_matched.csv')
MODEL

In [ ]:
fig, axes = plt.subplots(3, figsize=(8, 7), sharex=True, gridspec_kw={'hspace':0})

ax = axes[0]
ax.plot(oras5_t_sst, oras5_sst, color='#F54927', label='Temperature')

ax2 = ax.twinx()
ax2.plot(oras5_t, oras5_sss, color='#46bd9b', label='Salinity')

ax2.set_xlim(np.datetime64('1995-01-01') , np.datetime64('2006-01-01'))
ax2.set_ylabel('Sea Surface'+'\n'+'Salinity [ppt]')
ax.set_ylabel('Sea Surface'+'\n'+'Temperature [°C]')
ax.legend(loc='lower left')
ax2.legend(loc='lower right')
ax.grid()

ax = axes[2]
ax.plot(df_sorted['age_dt'], df_sorted['d13c'], '.-', color='green')
ax.set_ylim(-3.2, -1)
ax.set_ylabel(r'$\delta^{13}C$')
ax.grid()
ax.yaxis.tick_right()
ax.yaxis.set_label_position("right") # Also moves the label position

ax = axes[1]
#ax.plot(df_sorted['age_dt'], df_sorted['d18o'], '.-', color='cyan')
ax.plot(df_sorted['age_dt'], df_sorted['d18o_pdb'], '.-')
ax.set_ylabel(r'$\delta^{18}O$ [PDB]')
ax.set_ylim(-5.8,-4.8)
ax.grid()

#fig.savefig('ovrelook.png')
#fig.savefig('ovrelook.pdf')

In [ ]:
################################################
                                               #
#            a_1 dSST + a_2 dSSS = d18O        #
                                               #
#               a_1 = -0.22+-0.02              #
#               a_2 =  0.45+-0.028             #
                                               #
################################################
## see:
## https://agupubs.onlinelibrary.wiley.com/doi/full/10.1029/2011GL048224



#δ18OVPDB=0.97001×δ18OVSMOW−29.99\delta ^{18}O_{VPDB}=0.97001\times \delta ^{18}O_{VSMOW}-29.99
#
# see:
# https://isolab.ess.washington.edu/resources/vpdb-vsmow.php#:~:text=Introduction,O%20to%20a%20VSMOW%20value.

In [ ]:
#df_sorted['d18o_pdb'] = 0.97001*df_sorted['d18o']-29.99

In [ ]:
a_1 = -0.22
a_2 = 0.45
#a_2 = 0.8

synth_d18o = a_1*(oras5_sst- np.mean(oras5_sst)) + a_2*(oras5_sss- np.mean(oras5_sss))
synth_d18o = synth_d18o - synth_d18o.mean()

climate['synthetic_d18o'] = synth_d18o

a_1 = -0.22
#a_2 = 0.45
a_2 = 2

synth_d18o2 = a_1*(oras5_sst- np.mean(oras5_sst)) + a_2*(oras5_sss- np.mean(oras5_sss))
synth_d18o2 = synth_d18o2  - synth_d18o2.mean()
climate['synthetic_d18o_2'] = synth_d18o2

climate

In [ ]:
df_sorted['d18o_pdb_no_tectonic']

In [ ]:
fig, ax = plt.subplots(figsize=(8,4))

ax.plot(oras5_t, synth_d18o, color='cyan', label=r'$\delta^{18}$O based on climate data')
ax.plot(oras5_t, synth_d18o2, color='red', label=r'$\delta^{18}$O salinitiy dominated')

#ax.plot(df_sorted['age_dt'], df_sorted['d18o_pdb'], label=r'$\delta^{18}$O PDB from coral')
ax.plot(df_sorted['age_dt'], df_sorted['d18o_pdb_no_tectonic']*2, label=r'$\delta^{18}$O PDB from coral')
#ax.plot(df_sorted['age_dt'], (df_sorted['d18o_pdb']+5.2)*3, label=r'$\delta^{18}$O PDB from coral')
ax.set_xlim(np.datetime64('1980-01-01') , np.datetime64('2004-01-01'))
ax.set_ylim(-2,2)
ax.legend()



#fig.savefig('tsa_d18o_climate_data.png')

In [ ]:
fig, axes = plt.subplots(2,figsize=(8,6), sharex=True, gridspec_kw={'hspace':0})

ax = axes[0]
ax.plot(oras5_t, synth_d18o, color='cyan', label=r'$\delta^{18}$O based on climate data')

#ax.plot(df_sorted['age_dt'], df_sorted['d18o_pdb'], label=r'$\delta^{18}$O PDB from coral')
ax.plot(df_sorted['age_dt'], (df_sorted['d18o_pdb']+5.2)*3, label=r'$\delta^{18}$O PDB from coral')
ax.set_xlim(np.datetime64('1980-01-01') , np.datetime64('2004-01-01'))
ax.set_ylim(-2,2)
ax.legend()
ax.set_ylabel(r'$\Delta \delta^{18}O$')

ax = axes[1]

ax.plot(oras5_t, synth_d18o2, color='red', label=r'$\delta^{18}$O salinitiy dominated')
ax.plot(df_sorted['age_dt'], (df_sorted['d18o_pdb']+5.2)*3)
ax.set_xlim(np.datetime64('1980-01-01') , np.datetime64('2004-01-01'))
ax.set_ylim(-2,2)
ax.legend()
ax.set_ylabel(r'$\Delta \delta^{18}O$')

fig.savefig('tsa_d18o_climate_data2.png')

In [ ]:
climate_month = climate.groupby(climate.index.month).mean()


fig, axes = plt.subplots(1,2, figsize=(12,4))

ax = axes[0]
ax.plot(np.arange(1, 13), climate_month['synthetic_d18o'], '.-', label=r'synthetic $\delta^{18}$O')
ax.set_ylabel(r'$\delta^{18}$O')
ax.set_xlabel('Months')
ax.set_xticks(np.arange(1,13))
ax.set_title(r'Climate Average for $\delta^{18}$O')
ax.legend()
#fig.savefig('climate_average.pdf')
ax.set_xticks(np.arange(1, 13))

ax = axes[1]
ax.plot(np.arange(1, 13), climate_month['SSS'], '^-', color='#46bd9b', label='SSS')
ax2 = ax.twinx()
ax2.plot(np.arange(1, 13), climate_month['SST'], 'v-', color='#F54927', label='SST')
ax.set_xticks(np.arange(1, 13))
ax.legend(loc='lower right')
ax2.legend(loc='lower left')


In [ ]:

MODEL['d18o_ma'] = MODEL['d18o'].rolling(window_size, center=True).mean()
MODEL

In [ ]:
fred

In [ ]:
a_1 = -0.22
#a_2 = 0.45
a_2 = 0.8

synth_d18o = a_1*(oras5_sst- np.mean(oras5_sst)) + a_2*(oras5_sss- np.mean(oras5_sss))



fig = make_subplots(specs=[[{"secondary_y": True}]])

#fig.add_trace(go.Scatter(x=df_sorted['age_dt'], y=df_sorted['d18o']-df_sorted['d18o'].mean(),  mode="markers", marker_color="#3492eb",  name='d18O', text=df_sorted['Name']),secondary_y=False)
#fig.add_trace(go.Scatter(x=df_resampled.index.values, y=df_resampled['d18o_ma']-df_resampled['d18o_ma'].mean(), mode="lines", line_color='#134b80', name='average'), secondary_y=False)
#fig.add_trace(go.Scatter(x=df_resampled.index.values, y=df_resampled['d18o_ma2']-df_resampled['d18o_ma2'].mean(), mode="lines", line_color='navy', name='average'), secondary_y=False)

#fig.add_trace(go.Scatter(x=df_sorted['age_dt'], y=df_sorted['d18o_no_tectonic'] ,  mode="markers", marker_color="#3492eb",  name='d18O', text=df_sorted['Name']),secondary_y=False)
#fig.add_trace(go.Scatter(x=df_sorted['age_dt'], y=df_sorted['d18o_notec_ma'] , mode="lines", line_color='#134b80', name='average'), secondary_y=False)

#fig.add_trace(go.Scatter(x=MODEL['age_absolute'], y=MODEL['d18o'] , mode="markers", line_color='#3492eb', name='raw'), secondary_y=False)
#fig.add_trace(go.Scatter(x=MODEL['age_absolute'], y=MODEL['d18o_ma'] , mode="lines", line_color='#134b80', name='average'), secondary_y=False)

fig.add_trace(go.Scatter(x=fred['decimal_year'], y=fred['d18o_pdb'] , mode="markers", line_color='#3492eb', name='raw'), secondary_y=False)
fig.add_trace(go.Scatter(x=fred['decimal_year'], y=fred['d18o_ma'] , mode="lines", line_color='#134b80', name='average'), secondary_y=False)


fig.add_trace(go.Scatter(x=climate['age_absolute'], y= climate['synthetic_d18o'] , mode="lines", line_color='#F54927', name='synthetic d18o', line_width=1), secondary_y=True)



fig.update_layout(
    yaxis2=dict(title="Secondary Y-axis Title", overlaying="y", side="right", gridcolor='lightgray', showgrid=False)) # Grid for secondary Y-axis)
#fig.update_layout(
#    xaxis2=dict(title="Secondary X-axis Title", overlaying="x", side="top", gridcolor='lightgray', showgrid=False)) # Grid for secondary Y-axis)

fig.update_layout(width=1200, height=600, title_text="d18O observed and d18O synthetic (climate)", template='simple_white')
# Set x-axis title
fig.update_xaxes(title_text="Year")

# Set y-axes titles
fig.update_yaxes(title_text= r'$\delta^{18}$ O Observerd', secondary_y=False)
fig.update_yaxes(title_text= r'$\delta^{18}$ O Synthetic', secondary_y=True)

fig.show()
#fig.write_html('d18o_syntetic-d18O-CONSTANT-TEMP.html')


In [ ]:
a_1 = -0.22
#a_2 = 0.45
a_2 = 0.8

synth_d18o = a_1*(oras5_sst- np.mean(oras5_sst)) + a_2*(oras5_sss- np.mean(oras5_sss))



fig = make_subplots(specs=[[{"secondary_y": True}]])

#fig.add_trace(go.Scatter(x=df_sorted['age_dt'], y=df_sorted['d18o']-df_sorted['d18o'].mean(),  mode="markers", marker_color="#3492eb",  name='d18O', text=df_sorted['Name']),secondary_y=False)
#fig.add_trace(go.Scatter(x=df_resampled.index.values, y=df_resampled['d18o_ma']-df_resampled['d18o_ma'].mean(), mode="lines", line_color='#134b80', name='average'), secondary_y=False)
#fig.add_trace(go.Scatter(x=df_resampled.index.values, y=df_resampled['d18o_ma2']-df_resampled['d18o_ma2'].mean(), mode="lines", line_color='navy', name='average'), secondary_y=False)

#fig.add_trace(go.Scatter(x=df_sorted['age_dt'], y=df_sorted['d18o_no_tectonic'] ,  mode="markers", marker_color="#3492eb",  name='d18O', text=df_sorted['Name']),secondary_y=False)
#fig.add_trace(go.Scatter(x=df_sorted['age_dt'], y=df_sorted['d18o_notec_ma'] , mode="lines", line_color='#134b80', name='average'), secondary_y=False)

fig.add_trace(go.Scatter(x=MODEL['age_absolute'], y=MODEL['d18o'] , mode="markers", line_color='#3492eb', name='raw'), secondary_y=False)
fig.add_trace(go.Scatter(x=MODEL['age_absolute'], y=MODEL['d18o_ma'] , mode="lines", line_color='#134b80', name='average'), secondary_y=False)



fig.add_trace(go.Scatter(x=climate['age_absolute'], y= climate['synthetic_d18o'] , mode="lines", line_color='#F54927', name='synthetic d18o', line_width=1), secondary_y=True)



fig.update_layout(
    yaxis2=dict(title="Secondary Y-axis Title", overlaying="y", side="right", gridcolor='lightgray', showgrid=False)) # Grid for secondary Y-axis)
#fig.update_layout(
#    xaxis2=dict(title="Secondary X-axis Title", overlaying="x", side="top", gridcolor='lightgray', showgrid=False)) # Grid for secondary Y-axis)

fig.update_layout(width=1200, height=600, title_text="d18O observed and d18O synthetic (climate)", template='simple_white')
# Set x-axis title
fig.update_xaxes(title_text="Year")

# Set y-axes titles
fig.update_yaxes(title_text= r'$\delta^{18}$ O Observerd', secondary_y=False)
fig.update_yaxes(title_text= r'$\delta^{18}$ O Synthetic', secondary_y=True)

fig.show()
#fig.write_html('d18o_syntetic-d18O-CONSTANT-TEMP.html')


In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# --- your synth signal ---
a_1 = -0.22
a_2 = 0.8
synth_d18o = a_1*(oras5_sst - np.mean(oras5_sst)) + a_2*(oras5_sss - np.mean(oras5_sss))

# --- base figure ---
fig = make_subplots(specs=[[{"secondary_y": True}]])

# observed
fig.add_trace(
    go.Scatter(
        x=df_sorted['age_dt'],
        y=df_sorted['d18o_no_tectonic'],
        mode="markers",
        marker_color="#3492eb",
        name='d18O',
        text=df_sorted['Name']
    ),
    secondary_y=False
)

fig.add_trace(
    go.Scatter(
        x=df_sorted['age_dt'],
        y=df_sorted['d18o_notec_ma'],
        mode="lines",
        line_color="#134b80",
        name='average'
    ),
    secondary_y=False
)

# --- create multiple shifted versions of the synthetic trace ---
# Choose shift range + step (in DAYS). Adjust to your use case.
shift_days = np.arange(-30*12*50, 30*12*50, 30)   # +/- 1 year in 7-day steps

# Add one synthetic trace per shift; show only the first initially
for i, sd in enumerate(shift_days):
    fig.add_trace(
        go.Scatter(
            x=oras5_t + np.timedelta64(int(sd), "D"),
            y=synth_d18o,
            mode="lines",
            line_color="#F54927",
            name="synthetic d18o",
            line_width=1,
            visible=(i == 0),
            showlegend=(i == 0)  # keep legend clean
        ),
        secondary_y=True
    )

# --- slider that toggles which shifted synthetic trace is visible ---
n_base = 2  # first two traces are observed
n_shift = len(shift_days)

steps = []
for i, sd in enumerate(shift_days):
    vis = [True]*n_base + [False]*n_shift
    vis[n_base + i] = True
    steps.append(
        dict(
            method="update",
            label=f"{sd:+d}d",
            args=[
                {"visible": vis},
                {"title": f"d18O observed and synthetic (ORAS5 shifted by {sd:+d} days)"}
            ],
        )
    )

fig.update_layout(
    sliders=[dict(
        active=0,
        currentvalue={"prefix": "x-shift (synthetic): "},
        pad={"t": 50},
        steps=steps
    )]
)

fig.update_layout(
    yaxis2=dict(title="Secondary Y-axis Title", overlaying="y", side="right", showgrid=False),
    width=1800,
    height=100,
    title_text="d18O observed and d18O synthetic (climate)",
    template="simple_white"
)

fig.update_xaxes(title_text="Year")
fig.update_yaxes(title_text=r'$\delta^{18}$ O Observed', secondary_y=False)
fig.update_yaxes(title_text=r'$\delta^{18}$ O Synthetic', secondary_y=True)

fig.show()


In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# --- synthetic (fixed x) ---
a_1 = -0.22
a_2 = 0.8
synth_d18o = a_1*(oras5_sst - np.mean(oras5_sst)) + a_2*(oras5_sss - np.mean(oras5_sss))

# --- figure ---
fig = make_subplots(specs=[[{"secondary_y": True}]])

# 1) FIXED ORAS5 trace (always visible)
fig.add_trace(
    go.Scatter(
        x=oras5_t,
        y=synth_d18o,
        mode="lines",
        line_color="#F54927",
        name="synthetic d18o (fixed)",
        line_width=1,
    ),
    secondary_y=True
)

# 2) Build many SHIFTED coral versions (toggle via slider)
# choose shift range + step in DAYS (adjust to your data resolution)
shift_days = np.arange(-365, 366, 7)  # +/- 1 year in 7-day steps

# If you want to shift these:
x_obs = df_sorted["age_dt"].values
y_pts = df_sorted["d18o_no_tectonic"].values     # or df_sorted["d18o"]
y_ma  = df_sorted["d18o_notec_ma"].values        # or df_sorted["d18o_ma"]

# Add two traces per shift: points + MA
start_idx = 1  # because trace 0 is the fixed ORAS5
for i, sd in enumerate(shift_days):
    x_shift = x_obs + np.timedelta64(int(sd), "D")

    fig.add_trace(
        go.Scatter(
            x=x_shift,
            y=y_pts,
            mode="markers",
            marker_color="#3492eb",
            name="d18O (shifted)",
            text=df_sorted["Name"],
            visible=(i == 0),
            showlegend=(i == 0)
        ),
        secondary_y=False
    )

    fig.add_trace(
        go.Scatter(
            x=x_shift,
            y=y_ma,
            mode="lines",
            line_color="#134b80",
            name="average (shifted)",
            visible=(i == 0),
            showlegend=(i == 0)
        ),
        secondary_y=False
    )

# 3) Slider: toggle which shifted coral pair is visible (ORAS5 stays on)
n_shifts = len(shift_days)
# visibility vector: [ORAS5] + [2 traces per shift]
steps = []
for i, sd in enumerate(shift_days):
    vis = [True] + [False]*(2*n_shifts)
    vis[start_idx + 2*i]     = True  # points for shift i
    vis[start_idx + 2*i + 1] = True  # MA for shift i

    steps.append(
        dict(
            method="update",
            label=f"{sd:+d}d",
            args=[
                {"visible": vis},
                {"title": f"d18O observed shifted by {sd:+d} days (ORAS5 fixed)"}
            ],
        )
    )

fig.update_layout(
    sliders=[dict(
        active=0,
        currentvalue={"prefix": "x-shift (observed): "},
        pad={"t": 50},
        steps=steps
    )],
    yaxis2=dict(title="Secondary Y-axis Title", overlaying="y", side="right", showgrid=False),
    width=2000,
    height=600,
    title_text="d18O observed (shiftable) and d18O synthetic (fixed)",
    template="simple_white"
)

fig.update_xaxes(title_text="Year")
fig.update_yaxes(title_text=r'$\delta^{18}$ O Observed', secondary_y=False)
fig.update_yaxes(title_text=r'$\delta^{18}$ O Synthetic', secondary_y=True)

fig.show()


In [ ]:
# Ensure df_resampled is sorted by date
df = df_resampled[['d13c','d18o', 'd13c_no_tectonic', 'd18o_no_tectonic', 'd13c_notec_ma']].copy()
df = df.sort_index()

# Target dates to align with
target_dates = pd.to_datetime(enso.loc[:685, 'Date'])

# Interpolate to get values at those dates
aligned = df.reindex(df.index.union(target_dates)).interpolate(method='time').loc[target_dates]
aligned['age_absolute'] = np.array([datetime_to_decimal_year(x) for x in pd.to_datetime(aligned.index)])
aligned['ENSO'] = enso.loc[:685, 'ENSO'].values
aligned_tidegauge = tide.reindex(tide.index.union(target_dates)).interpolate(method='time').loc[target_dates]
aligned = aligned.join(aligned_tidegauge['Sea Level'].dropna())
aligned_climate = climate.reindex(climate.index.union(target_dates)).interpolate(method='time').loc[target_dates]
aligned = aligned.join(aligned_climate[['SST', 'SSS', 'synthetic_d18o']].dropna())
correlation_matrix = aligned.loc['1969-01-01':'1975-01-01'].corr()
print(correlation_matrix)
#aligned.loc['2005-01-01':'2008-01-01']
aligned

In [ ]:
aligned.loc['1969-01-01':'1975-01-01'].corr()

In [ ]:
correlation_matrix = aligned.corr()

# Create a heatmap
fig = plt.figure(figsize=(7, 7))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, fmt=".2f")
plt.title("Correlation Matrix Heatmap")
plt.tight_layout()
plt.show()
fig.savefig('corr_matrix.png')

In [ ]:
#pd.DataFrame([oras5_t, synth_d18o]).T.to_csv('modeled_d18o.csv')
climate.to_csv('climate_df.csv')

In [ ]:
climate